# MLflow ML Trading Experiment Review

This notebook reviews completed Dagster-managed ML trading experiments. It does not build features, train models, score, or run backtests. Dagster owns execution; MLflow owns experiment tracking; `ArtifactStore` remains a local cache for artifact inspection.

In [1]:
from __future__ import annotations

import mlflow
import pandas as pd

from quant_orchestrator.research_tools import load_latest_experiment_artifacts
from quant_orchestrator.tracking import DEFAULT_TRACKING_URI

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 180)
pd.set_option('display.float_format', lambda value: f'{value:,.4f}')

MLFLOW_EXPERIMENT = 'ml_trading'
BASELINE_EXPERIMENT_NAME = 'gpu_rf_shared_book_1t_dagster_smoke'
AE_EXPERIMENT_NAME = 'gpu_rf_autoencoder_shared_book_1t_dagster_smoke'

mlflow.set_tracking_uri(DEFAULT_TRACKING_URI)
experiment = mlflow.get_experiment_by_name(MLFLOW_EXPERIMENT)
if experiment is None:
    raise RuntimeError(f'MLflow experiment not found: {MLFLOW_EXPERIMENT}')
print({'tracking_uri': DEFAULT_TRACKING_URI, 'experiment_id': experiment.experiment_id, 'experiment_name': experiment.name})

{'tracking_uri': 'sqlite:///artifacts/mlflow/mlflow.db', 'experiment_id': '1', 'experiment_name': 'ml_trading'}


In [2]:
runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=['start_time DESC'],
    max_results=100,
)
run_cols = [
    'tags.quant_orchestrator.experiment_name',
    'tags.quant_orchestrator.mode',
    'tags.quant_orchestrator.provider',
    'metrics.best_sharpe',
    'metrics.best_total_return',
    'metrics.best_max_drawdown',
    'metrics.trained_models',
    'metrics.strategy_sources',
    'metrics.elapsed_seconds',
    'status',
    'start_time',
    'run_id',
]
run_cols = [col for col in run_cols if col in runs.columns]
mlflow_runs = runs[run_cols].copy()
display(mlflow_runs)

,tags.quant_orchestrator.experiment_name,tags.quant_orchestrator.mode,tags.quant_orchestrator.provider,metrics.best_sharpe,metrics.best_total_return,metrics.best_max_drawdown,metrics.trained_models,metrics.strategy_sources,metrics.elapsed_seconds,status,start_time,run_id
0,gpu_rf_autoencoder_shared_book_1t_dagster_smoke,classifier_ae,fmp,1.2959,5.4210,-0.3799,15.0000,16.0000,234.9811,FINISHED,2026-07-02 06:45:44.282000+00:00,e4e799db6dd2448c8c71c2d4dcde2cb0
1,gpu_rf_shared_book_1t_dagster_smoke,classifier,fmp,1.5984,8.1790,-0.3022,15.0000,16.0000,226.0861,FINISHED,2026-07-02 06:41:29.146000+00:00,06ed53bde70841d79d929cdeacce003f


In [3]:
baseline = load_latest_experiment_artifacts(BASELINE_EXPERIMENT_NAME)
ae = load_latest_experiment_artifacts(AE_EXPERIMENT_NAME)

baseline_summary = baseline['backtest_summary'].assign(experiment_name=BASELINE_EXPERIMENT_NAME)
ae_summary = ae['backtest_summary'].assign(experiment_name=AE_EXPERIMENT_NAME)
comparison = pd.concat([baseline_summary, ae_summary], ignore_index=True)

cols = [
    'experiment_name', 'strategy_source', 'source', 'family', 'variant', 'top_k',
    'total_return', 'sharpe', 'max_drawdown', 'avg_gross_exposure', 'avg_net_exposure',
    'trades', 'signal_events',
]
cols = [col for col in cols if col in comparison.columns]
display(comparison[cols].sort_values(['sharpe', 'total_return'], ascending=False).head(25))

,experiment_name,strategy_source,source,family,variant,top_k,total_return,sharpe,max_drawdown,avg_gross_exposure,avg_net_exposure,trades,signal_events
168,gpu_rf_shared_book_1t_dagster_smoke,fmp.fmp_daily_mcap_yield,fmp,fmp_daily_mcap_yield,long_only,5,8.1790,1.5984,-0.3022,0.8942,0.8942,6312,441
30,gpu_rf_shared_book_1t_dagster_smoke,financetoolkit.ft_growth_cash,financetoolkit,ft_growth_cash,long_short,20,2.8823,1.3953,-0.2338,0.6500,0.6396,16885,23
31,gpu_rf_shared_book_1t_dagster_smoke,financetoolkit.ft_growth_cash,financetoolkit,ft_growth_cash,long_short,40,1.0046,1.3881,-0.1203,0.3250,0.3198,13661,23
17,gpu_rf_shared_book_1t_dagster_smoke,financetoolkit.ft_growth_balance,financetoolkit,ft_growth_balance,long_short,10,5.2266,1.3796,-0.2792,1.0000,0.9936,13709,12
18,gpu_rf_shared_book_1t_dagster_smoke,financetoolkit.ft_growth_balance,financetoolkit,ft_growth_balance,long_short,20,2.9116,1.3685,-0.2337,0.6500,0.6450,16927,17
54,gpu_rf_shared_book_1t_dagster_smoke,financetoolkit.ft_ratios_efficiency,financetoolkit,ft_ratios_efficiency,long_short,20,2.6490,1.3659,-0.2337,0.6500,0.6157,16914,33
102,gpu_rf_shared_book_1t_dagster_smoke,financetoolkit.ft_ratios_valuation,financetoolkit,ft_ratios_valuation,long_short,20,2.7878,1.3639,-0.2271,0.6500,0.6271,16905,25
19,gpu_rf_shared_book_1t_dagster_smoke,financetoolkit.ft_growth_balance,financetoolkit,ft_growth_balance,long_short,40,1.0129,1.3601,-0.1205,0.3250,0.3225,13634,17
55,gpu_rf_shared_book_1t_dagster_smoke,financetoolkit.ft_ratios_efficiency,financetoolkit,ft_ratios_efficiency,long_short,40,0.9425,1.3598,-0.1203,0.3250,0.3079,13600,33
103,gpu_rf_shared_book_1t_dagster_smoke,financetoolkit.ft_ratios_valuation,financetoolkit,ft_ratios_valuation,long_short,40,0.9791,1.3556,-0.1171,0.3250,0.3135,13562,25


In [4]:
best_by_experiment = (
    comparison[cols]
    .sort_values(['experiment_name', 'sharpe', 'total_return'], ascending=[True, False, False])
    .groupby('experiment_name', as_index=False)
    .head(1)
    .sort_values(['sharpe', 'total_return'], ascending=False)
    .reset_index(drop=True)
)
display(best_by_experiment)

if set(best_by_experiment['experiment_name']) >= {BASELINE_EXPERIMENT_NAME, AE_EXPERIMENT_NAME}:
    base = best_by_experiment.loc[best_by_experiment['experiment_name'].eq(BASELINE_EXPERIMENT_NAME)].iloc[0]
    ae_best = best_by_experiment.loc[best_by_experiment['experiment_name'].eq(AE_EXPERIMENT_NAME)].iloc[0]
    deltas = {
        'ae_minus_classifier_sharpe': float(ae_best['sharpe']) - float(base['sharpe']),
        'ae_minus_classifier_total_return': float(ae_best['total_return']) - float(base['total_return']),
        'ae_minus_classifier_max_drawdown': float(ae_best['max_drawdown']) - float(base['max_drawdown']),
    }
    print(deltas)

,experiment_name,strategy_source,source,family,variant,top_k,total_return,sharpe,max_drawdown,avg_gross_exposure,avg_net_exposure,trades,signal_events
0,gpu_rf_shared_book_1t_dagster_smoke,fmp.fmp_daily_mcap_yield,fmp,fmp_daily_mcap_yield,long_only,5,8.1790,1.5984,-0.3022,0.8942,0.8942,6312,441
1,gpu_rf_autoencoder_shared_book_1t_dagster_smoke,fmp.fmp_daily_mcap_yield,fmp,fmp_daily_mcap_yield,long_only,5,5.4210,1.2959,-0.3799,0.8892,0.8892,6195,148


{'ae_minus_classifier_sharpe': -0.3025, 'ae_minus_classifier_total_return': -2.758, 'ae_minus_classifier_max_drawdown': -0.07769999999999999}


In [5]:
from IPython.display import Markdown, display

analysis_lines = [
    '## Written Analysis',
    '',
    f'- MLflow experiment: `{MLFLOW_EXPERIMENT}`.',
    f'- Baseline artifact: `{BASELINE_EXPERIMENT_NAME}`.',
    f'- AE artifact: `{AE_EXPERIMENT_NAME}`.',
    f'- MLflow runs loaded: {len(mlflow_runs)}.',
]
if not best_by_experiment.empty:
    for row in best_by_experiment.itertuples(index=False):
        analysis_lines.append(
            f'- Best `{row.experiment_name}` row: {row.strategy_source} / {row.variant} top_k={int(row.top_k)}; '
            f'total_return={row.total_return:.2%}, sharpe={row.sharpe:.2f}, max_drawdown={row.max_drawdown:.2%}.'
        )
analysis_lines.extend([
    '',
    'Interpretation:',
    '- This notebook is intentionally read-only with respect to training and backtesting.',
    '- Use Dagster for running experiments and MLflow for comparing tracked runs.',
    '- Use ArtifactStore only to load detailed local tables that were logged alongside the MLflow run.',
])
analysis_markdown = '\n'.join(analysis_lines)
display(Markdown(analysis_markdown))

## Written Analysis

- MLflow experiment: `ml_trading`.
- Baseline artifact: `gpu_rf_shared_book_1t_dagster_smoke`.
- AE artifact: `gpu_rf_autoencoder_shared_book_1t_dagster_smoke`.
- MLflow runs loaded: 2.
- Best `gpu_rf_shared_book_1t_dagster_smoke` row: fmp.fmp_daily_mcap_yield / long_only top_k=5; total_return=817.90%, sharpe=1.60, max_drawdown=-30.22%.
- Best `gpu_rf_autoencoder_shared_book_1t_dagster_smoke` row: fmp.fmp_daily_mcap_yield / long_only top_k=5; total_return=542.10%, sharpe=1.30, max_drawdown=-37.99%.

Interpretation:
- This notebook is intentionally read-only with respect to training and backtesting.
- Use Dagster for running experiments and MLflow for comparing tracked runs.
- Use ArtifactStore only to load detailed local tables that were logged alongside the MLflow run.

## Written Analysis

- MLflow experiment: `ml_trading`.
- Baseline artifact: `gpu_rf_shared_book_1t_dagster_smoke`.
- AE artifact: `gpu_rf_autoencoder_shared_book_1t_dagster_smoke`.
- MLflow runs loaded: 2.
- Best `gpu_rf_shared_book_1t_dagster_smoke` row: fmp.fmp_daily_mcap_yield / long_only top_k=5; total_return=817.90%, sharpe=1.60, max_drawdown=-30.22%.
- Best `gpu_rf_autoencoder_shared_book_1t_dagster_smoke` row: fmp.fmp_daily_mcap_yield / long_only top_k=5; total_return=542.10%, sharpe=1.30, max_drawdown=-37.99%.

Interpretation:
- This notebook is intentionally read-only with respect to training and backtesting.
- Use Dagster for running experiments and MLflow for comparing tracked runs.
- Use ArtifactStore only to load detailed local tables that were logged alongside the MLflow run.